# MIMIC-III query API tutorial

This notebook shows examples of how to use the cyclops.query API on [MIMIC-III v1.4](https://physionet.org/content/mimiciii/1.4/).

Each query is limit to 100 rows (for quick results).

* First, setup the MIMIC-III database according to the instructions in [mimic-code](https://github.com/MIT-LCP/mimic-code/tree/main/mimic-iii/buildmimic/postgres).
* The database is assumed to be hosted using postgres. Update the config parameters such as username and password, passed to `MIMICIIIQuerier` accordingly.

## Imports and instantiate `MIMICIIIQuerier`

In [1]:
"""MIMICIII query API tutorial."""

import cyclops.query.ops as qo
from cyclops.query import MIMICIIIQuerier


querier = MIMICIIIQuerier(
    dbms="postgresql",
    port=5432,
    host="localhost",
    database="mimiciii",
    user="postgres",
    password="pwd",
)
# List all custom table methods.
querier.list_custom_tables()

/home/amritk/.cache/pypoetry/virtualenvs/pycyclops-wIzUAwxh-py3.9/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2023-10-10 09:07:14,379 INFO cyclops.query.orm - Database setup, ready to run queries!


['chartevents', 'diagnoses', 'labevents']

## Example 1. Get all male patients with a mortality outcome.

In [2]:
ops = qo.Sequential(
    qo.ConditionEquals("expire_flag", 1),
    qo.ConditionEquals("gender", "M"),
)
patients = querier.mimiciii.patients()
patients = patients.ops(ops).run(limit=100)
print(f"{len(patients)} rows extracted!")

2023-10-10 09:07:20,955 INFO cyclops.query.orm - Query returned successfully!


2023-10-10 09:07:20,956 INFO cyclops.utils.profile - Finished executing function run_query in 0.021412 s


100 rows extracted!


## Example 2. Get all female patient encounters with diagnoses (`gastroenteritis` in ICD-9 long title).

In [3]:
patients = querier.mimiciii.patients()
patients = patients.ops(qo.ConditionEquals("gender", "F"))
admissions = querier.mimiciii.admissions()
patient_admissions = patients.join(
    join_table=admissions,
    on="subject_id",
)
diagnoses = querier.diagnoses()
diagnoses = diagnoses.ops(qo.ConditionSubstring("long_title", "gastroenteritis"))
patient_admissions_diagnoses = patient_admissions.join(
    join_table=diagnoses,
    on=["subject_id", "hadm_id"],
).run(limit=100)
print(f"{len(patient_admissions_diagnoses)} rows extracted!")

2023-10-10 09:07:21,071 INFO cyclops.query.orm - Query returned successfully!


2023-10-10 09:07:21,072 INFO cyclops.utils.profile - Finished executing function run_query in 0.086344 s


100 rows extracted!


## Example 3. Get potassium lab tests for female patients.

In [4]:
patients = querier.mimiciii.patients()
patients = patients.ops(qo.ConditionEquals("gender", "F"))
labs = querier.labevents()
labs = labs.ops(qo.ConditionEquals("label", "potassium"))
patient_labs = patients.join(labs, on="subject_id").run(limit=100)
print(f"{len(patient_labs)} rows extracted!")

2023-10-10 09:07:21,137 INFO cyclops.query.orm - Query returned successfully!


2023-10-10 09:07:21,139 INFO cyclops.utils.profile - Finished executing function run_query in 0.031545 s


100 rows extracted!


## Example 4. Get AaDO2 carevue chart events for male patients that have a `valuenum` of less than 20.

In [5]:
chartevents_ops = qo.Sequential(
    qo.ConditionEquals("dbsource", "carevue"),
    qo.ConditionEquals("label", "AaDO2"),
    qo.ConditionLessThan("valuenum", 20),
)
patients = querier.mimiciii.patients()
patients = patients.ops(qo.ConditionEquals("gender", "M"))
chart_events = querier.chartevents()
chart_events = chart_events.ops(chartevents_ops)
patient_chart_events = patients.join(chart_events, on="subject_id").run(limit=100)
print(f"{len(patient_chart_events)} rows extracted!")

2023-10-10 09:08:35,607 INFO cyclops.query.orm - Query returned successfully!


2023-10-10 09:08:35,613 INFO cyclops.utils.profile - Finished executing function run_query in 74.437042 s


6 rows extracted!
